# 05. Predict — загрузить лучший чекпоинт → submission

Этот ноутбук не обучает модели.
Он принимает `best.pt` от любого из 4 основных ноутбуков и генерирует submission.csv.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Idempotent data download. Skip if notebooks/data/ already populated.
from pathlib import Path
import zipfile

DATA_ROOT = Path("notebooks/data")
_ZIP_URL = "https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link"
_ZIP_PATH = DATA_ROOT / "data.zip"

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    import gdown

    gdown.download(url=_ZIP_URL, output=str(_ZIP_PATH), quiet=False, fuzzy=True)
    with zipfile.ZipFile(_ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)
    print(f"Extracted data to {DATA_ROOT}")
else:
    print(f"Data already present at {DATA_ROOT} — skipping download.")


## Env hardening

In [ ]:
# Env hardening — must run BEFORE `import torch`.
import os

# Снижает фрагментацию VRAM — критично для torch.compile + переменных T.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

# cudnn.benchmark=False — переменные T после padding, autotune только мешает.
torch.backends.cudnn.benchmark = False
# TF32 для matmul — ускоряет fp32 операции на Ampere+ без потери CER.
torch.set_float32_matmul_precision("high")


## Пути (заполните вручную)

Задай абсолютные пути под свою среду. Все `FILL_ME_IN` должны быть реальными путями к train/dev/test и директории чекпоинтов.

In [ ]:
from pathlib import Path

# DATA_ROOT was defined in the download cell above.
TRAIN_ROOT = DATA_ROOT / "data" / "train"
DEV_ROOT = DATA_ROOT / "data" / "dev"
TEST_ROOT: Path | None = DATA_ROOT / "data" / "test"
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = Path("checkpoints")

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"

print(f"train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")


Укажи путь к `best.pt` любой из четырёх основных моделей (смотри `meta.json` рядом с ним).

In [ ]:
from pathlib import Path
import json

# Set CKPT to a real checkpoint path before running.
# Example: CKPT = Path("checkpoints/01_quartznet/trial_03/best.pt")
CKPT: Path | None = None

if CKPT is None:
    raise RuntimeError(
        "Set CKPT to a valid checkpoint path before running. "
        "Run one of the training notebooks (01 / 02 / 03 / 04 / 01a / 01b / 01c) first."
    )
if not CKPT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT}")

meta = json.loads((CKPT.parent / "meta.json").read_text())
print("Arch:", meta["arch"])
print("val_cer:", meta.get("val_cer", "N/A"))
print("hparams:", json.dumps(meta.get("hparams", {}), indent=2, default=str))


## Собрать модель по arch + hparams из meta

Диспатч по `meta["arch"]`: создаём нужный класс модели и загружаем веса.

In [ ]:
import torch
from gp1.train.checkpoint import load_checkpoint

arch = meta["arch"]
hp = meta.get("hparams", {})

if arch == "quartznet_10x4":
    from gp1.models.quartznet import QuartzNet10x4
    from gp1.text.vocab import CharVocab

    vocab = CharVocab()
    model = QuartzNet10x4(
        vocab_size=vocab.vocab_size,
        d_model=hp.get("d_model", 256),
        dropout=hp.get("dropout", 0.1),
        subsample_factor=hp.get("subsample_factor", 2),
    ).to(device)

elif arch == "crdnn":
    from gp1.models.crdnn import CRDNN
    from gp1.text.vocab import CharVocab

    vocab = CharVocab()
    model = CRDNN(
        vocab_size=vocab.vocab_size,
        d_cnn=hp.get("d_cnn", 64),
        rnn_hidden=hp.get("rnn_hidden", 256),
        rnn_layers=hp.get("rnn_layers", 2),
        dropout=hp.get("dropout", 0.15),
        subsample_factor=hp.get("subsample_factor", 1),
    ).to(device)

elif arch == "efficient_conformer":
    from gp1.models.efficient_conformer import EfficientConformer
    from gp1.text.vocab import CharVocab

    vocab = CharVocab()
    d_stages = hp.get("d_model_stages", [96, 128, 128])
    n_stages = hp.get("n_blocks_per_stage", [4, 4, 4])
    model = EfficientConformer(
        vocab_size=vocab.vocab_size,
        d_model_stages=tuple(d_stages),
        n_blocks_per_stage=tuple(n_stages),
        n_heads=hp.get("n_heads", 4),
        ff_ratio=hp.get("ff_ratio", 4),
        conv_kernel=hp.get("conv_kernel", 15),
        dropout=hp.get("dropout", 0.1),
    ).to(device)

elif arch == "fast_conformer_bpe":
    from gp1.models.fast_conformer_bpe import FastConformerBPE
    from gp1.text.vocab_bpe import BPEVocab

    bpe_model_path = CKPT_ROOT.parent / "bpe_models" / f"bpe_{hp.get('bpe_vocab_size', 256)}.model"
    vocab = BPEVocab(bpe_model_path)
    model = FastConformerBPE(
        vocab_size=vocab.vocab_size,
        d_model=hp.get("d_model", 96),
        n_blocks=hp.get("n_blocks", 16),
        n_heads=hp.get("n_heads", 4),
        ff_ratio=hp.get("ff_ratio", 4),
        conv_kernel=hp.get("conv_kernel", 9),
        dropout=hp.get("dropout", 0.1),
        subsample_factor=hp.get("subsample_factor", 4),
    ).to(device)

else:
    raise ValueError(f"Unknown arch: {arch}")

load_checkpoint(CKPT, model)
model.eval()
print(f"Loaded {arch} from {CKPT}")
torch.backends.cudnn.benchmark = True
torch.set_num_threads(1)

## Инференс на test + submission

Запускаем greedy decode на всех тестовых файлах и пишем `submission.csv`.

In [ ]:
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.decoding.greedy import greedy_decode
from gp1.text.denormalize import words_to_digits
from gp1.submit.inference_utils import build_test_dataloader, write_submission

if TEST_ROOT is None:
    print("No test_root configured — set GP1_TEST_ROOT or place data in data/test/")
else:
    hop_length = hp.get("hop_length", 160)
    mel_extractor = LogMelFilterBanks(
        n_fft=hp.get("n_fft", 512),
        samplerate=hp.get("samplerate", 16000),
        hop_length=hop_length,
        win_length=hp.get("win_length", 400),
        n_mels=hp.get("n_mels", 80),
    ).to(device)

    test_records = records_from_csv(TEST_ROOT / "test.csv", TEST_ROOT)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    all_hyps = []
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // hop_length + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_hyps.extend(hyps)

    pairs = [
        (rec.audio_path.name, words_to_digits(h))
        for rec, h in zip(test_records, all_hyps)
    ]
    submission_path = CKPT_ROOT / "submission.csv"
    write_submission(pairs, submission_path)
    print(f"Submission saved ({len(pairs)} rows):", submission_path)
